# Ajuste de columnas para la Práctica 4

Se genera `endireh_2021_renombrado.csv` a partir de `endireh_2021_limpio.csv`.
Además del renombrado y la reparación geografica, se recuperan de la fuente original unicamente los codigos validos de 0 a 9 de `anios_sin_convivencia_pareja` que quedaron nulos por el filtro previo.
El archivo limpio, el notebook de exploracion y los resultados de la Practica 3 se conservan.


## 1. Configuración y carga

Usar el entorno Python que hayas creado. Las rutas de archivos se importan de `config/rutas.py`.


In [35]:
from pathlib import Path
import hashlib
import json
import sys

import polars as pl

RUTA_PROYECTO = next(
    (ruta for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / "config" / "rutas.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("Abre el notebook dentro del proyecto.")
if str(RUTA_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RUTA_PROYECTO))

# Recarga las rutas si el kernel conserva una versión anterior del módulo.
import importlib
import config.rutas as rutas
importlib.reload(rutas)

from config.rutas import (
    RUTA_ENDIREH_CSV, RUTA_ENDIREH_LIMPIO_CSV, RUTA_ENDIREH_RENOMBRADO_CSV,
    RUTA_RESULTADOS_AJUSTE, RUTA_RECUPERACION_TIEMPOS_CSV, RUTA_RESUMEN_AJUSTE_JSON,
)

if RUTA_ENDIREH_LIMPIO_CSV.resolve() == RUTA_ENDIREH_RENOMBRADO_CSV.resolve():
    raise ValueError("Las rutas de entrada y salida deben ser distintas.")
if not RUTA_ENDIREH_LIMPIO_CSV.is_file():
    raise FileNotFoundError(f"No se encontró el CSV limpio: {RUTA_ENDIREH_LIMPIO_CSV}")

hash_entrada = hashlib.sha256(RUTA_ENDIREH_LIMPIO_CSV.read_bytes()).hexdigest()
df_limpio = pl.read_csv(RUTA_ENDIREH_LIMPIO_CSV)
print(f"Entrada: {RUTA_ENDIREH_LIMPIO_CSV.name}")
print(f"Registros: {df_limpio.height:,} | Columnas: {df_limpio.width}")


Entrada: endireh_2021_limpio.csv
Registros: 110,127 | Columnas: 24


## 2. Correspondencias de columnas

La tabla indica el nombre recibido, el nuevo nombre, la variable original inferida y el motivo del cambio.
Las demás columnas mantienen sus nombres. Las binarias conservan 1 = Sí y 2 = No; el indicador de violencia conserva 0/1.

Referencia: [Estructura de la base de datos ENDIREH 2021, INEGI](https://www.inegi.org.mx/contenidos/programas/endireh/2021/doc/endireh2021_fd.pdf). 


In [36]:
correspondencias = [
    ("nivel_escolaridad", "tipo_cuestionario", "T_INSTRUM", "A1-C2 identifica el instrumento segun situación conyugal, no escolaridad."),
    ("num_hijos", "tipo_union", "P4AB_1", "1: unión libre; 2: solo iglesia; 3: iglesia y civil; 4: solo civil. No es un conteo de hijos."),
    ("edad_primer_union", "anios_sin_convivencia_pareja", "P4AB_2", "Tiempo desde que la pareja o expareja dejó de vivir con la entrevistada, no edad."),
    ("pareja_trabaja_id", "entrevistada_trabaja_id", "P4_1", "Trabajo remunerado de la entrevistada: 1 Sí, 2 No."),
    ("pareja_trabaja_desc", "entrevistada_trabaja_desc", "P4_1", "Descripcion de la condición de trabajo remunerado de la entrevistada."),
    ("ingreso_pareja", "ingreso_laboral_entrevistada", "P4_2", "Monto que gana la entrevistada por su trabajo. La periodicidad no está disponible."),
    ("tiene_ahorros_id", "hogar_tiene_ahorros_id", "P4_12_3", "Algún miembro del hogar tiene ahorros; no implica titularidad de la entrevistada."),
    ("tiene_ahorros_desc", "hogar_tiene_ahorros_desc", "P4_12_3", "Descripcio de la tenencia de ahorros por algn miembro del hogar."),
    ("propietaria_vivienda_id", "vivienda_propiedad_miembro_hogar_id", "P4_12_4", "Algún miembro del hogar es propietario de la vivienda que habitan."),
    ("propietaria_vivienda_desc", "vivienda_propiedad_miembro_hogar_desc", "P4_12_4", "Descripcion de la propiedad de la vivienda por algún miembro del hogar."),
    ("estrato_socioeconomico", "estrato_sociodemografico_muestreo", "ESTRATO", "Estrato del area de muestreo; no se asigna una jerarquia econmica individual."),
]

# Convierte la lista de correspondencias en un DataFrame de Polars
tabla_correspondencias = pl.DataFrame(
    correspondencias,
    schema=["nombre_original", "nombre_nuevo", "variable_endireh", "justificacion"],
    orient="row",
)
columnas_renombradas = dict(zip(
    tabla_correspondencias["nombre_original"], tabla_correspondencias["nombre_nuevo"]
))

# Verifica que todas las columnas que se desean renombrar realmente existan en el DataFrame original.
faltantes = set(columnas_renombradas).difference(df_limpio.columns)
if faltantes:
    raise ValueError(f"Faltan columnas de origen: {sorted(faltantes)}")
if len(set(columnas_renombradas.values())) != len(columnas_renombradas):
    raise ValueError("Los nombres de destino deben ser únicos.")
# Comprueba que los nuevos nombres no correspondan a columnas que ya existen en el DataFrame original.
colisiones = set(columnas_renombradas.values()).intersection(df_limpio.columns)
if colisiones:
    raise ValueError(f"Los nombres de destino ya existen: {sorted(colisiones)}")
# Renombra las columnas utilizando las correspondencias
df_renombrado = df_limpio.rename(columnas_renombradas)
tabla_correspondencias


nombre_original,nombre_nuevo,variable_endireh,justificacion
str,str,str,str
"""nivel_escolaridad""","""tipo_cuestionario""","""T_INSTRUM""","""A1-C2 identifica el instrument…"
"""num_hijos""","""tipo_union""","""P4AB_1""","""1: unión libre; 2: solo iglesi…"
"""edad_primer_union""","""anios_sin_convivencia_pareja""","""P4AB_2""","""Tiempo desde que la pareja o e…"
"""pareja_trabaja_id""","""entrevistada_trabaja_id""","""P4_1""","""Trabajo remunerado de la entre…"
"""pareja_trabaja_desc""","""entrevistada_trabaja_desc""","""P4_1""","""Descripcion de la condición de…"
…,…,…,…
"""tiene_ahorros_id""","""hogar_tiene_ahorros_id""","""P4_12_3""","""Algún miembro del hogar tiene …"
"""tiene_ahorros_desc""","""hogar_tiene_ahorros_desc""","""P4_12_3""","""Descripcio de la tenencia de a…"
"""propietaria_vivienda_id""","""vivienda_propiedad_miembro_hog…","""P4_12_4""","""Algún miembro del hogar es pro…"


## 3. Correccion de nombres geograficos

Se repara el texto UTF-8 que se interpreto como Latin-1, úncamente cuando contiene indicios de codificacion dañada y la conversion es válida. 

In [37]:
def corregir_nombre_geografico(nombre):
    if nombre is None or not any(marca in nombre for marca in ("Ã", "Â")):
        return nombre
    try:
        return nombre.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return nombre

# Corige nombres de entidades y municipios, y registra cuantos valores cambiaron.
resumen_geografia = []
for columna in ("nom_entidad", "nom_municipio"):
    cambios = {}
    for nombre in df_renombrado[columna].drop_nulls().unique().sort().to_list():
        corregido = corregir_nombre_geografico(nombre)
        if corregido != nombre:
            cambios[nombre] = corregido
    resumen_geografia.append({
        "columna": columna,
        "nombres_distintos_corregidos": len(cambios),
        "registros_afectados": df_renombrado.filter(pl.col(columna).is_in(list(cambios))).height,
    })
    if cambios:
        df_renombrado = df_renombrado.with_columns(pl.col(columna).replace(cambios))

pl.DataFrame(resumen_geografia)


columna,nombres_distintos_corregidos,registros_afectados
str,i64,i64
"""nom_entidad""",7,24116
"""nom_municipio""",402,36781


## 4. Recuperar tiempos válidos de 0 a 9 años

En P3 se interpretó esta columna como edad y se conservaron solo valores de 10 a 97. Para P4 corregimos esa decision exclusivamente en el derivado. No es posible recuperar los valores a partir de los nulos del CSV limpio: se consulta la misma fuente del curso y se comprueba su correspondencia completa antes de recuperar.

Para P4AB_2, **0 indica menos de un año**, 1–97 son tiempos declarados, 98 indica «no se acuerda» y 99 «no especificado». Los codigos 98/99 y los blancos permanecen nulos; no se imputan tiempos. Universo: A2/B1/B2.

In [39]:
# Carga la fuente original para recuperar los valores 0–9 eliminados en P3.
if not RUTA_ENDIREH_CSV.is_file():
    raise FileNotFoundError(f"No se encontró la fuente original: {RUTA_ENDIREH_CSV}")

df_fuente = pl.read_csv(RUTA_ENDIREH_CSV, infer_schema_length=None)

# Normaliza la fuente igual que en P3.
columnas_texto = [
    c for c, tipo in df_fuente.schema.items()
    if tipo == pl.String
]

df_fuente_normalizada = (
    df_fuente
    .with_columns(
        pl.col(c).str.strip_chars().replace("", None)
        for c in columnas_texto
    )
    .with_columns(
        pl.col("edad_primer_union").cast(pl.Int64, strict=False),
        pl.col("num_hijos").cast(pl.Int64, strict=False),
        pl.col("sufrio_violencia_pareja").cast(pl.Int8),
        pl.col("anio_encuesta").cast(pl.Int16),
    )
)

# Comprueba que el CSV limpio corresponde a esta misma fuente.
valor = pl.col("edad_primer_union")

esperado_p3 = df_fuente_normalizada.with_columns(
    pl.when(valor.is_between(10, 97))
    .then(valor)
    .otherwise(None)
    .alias("edad_primer_union")
)

if not esperado_p3.equals(df_limpio):
    raise ValueError("El CSV limpio no corresponde con la fuente original.")

# Identifica los valores 0–9 que deben recuperarse.
mascara_recuperacion = (
    df_fuente_normalizada["edad_primer_union"]
    .is_between(0, 9)
    .fill_null(False)
)

# Verifica que pertenezcan al universo esperado.
if df_fuente_normalizada.filter(
    mascara_recuperacion
    & ~pl.col("nivel_escolaridad")
        .is_in(["A2", "B1", "B2"])
        .fill_null(False)
).height:
    raise ValueError("Hay valores 0–9 fuera del universo A2/B1/B2.")

# Recupera los valores.
antes = df_renombrado["anios_sin_convivencia_pareja"].drop_nulls().len()

df_renombrado = df_renombrado.with_columns(
    pl.when(mascara_recuperacion)
    .then(df_fuente_normalizada["edad_primer_union"])
    .otherwise(pl.col("anios_sin_convivencia_pareja"))
    .alias("anios_sin_convivencia_pareja")
)

despues = df_renombrado["anios_sin_convivencia_pareja"].drop_nulls().len()

print(f"Valores de 0 a 9 recuperados: {int(mascara_recuperacion.sum()):,}")
print(f"Disponibles antes: {antes:,}")
print(f"Disponibles después: {despues:,}")

Valores de 0 a 9 recuperados: 14,252
Disponibles antes: 23,677
Disponibles después: 23,677


## 5. Guardar y comprobar el CSV nuevo

La salida es independiente del archivo limpio de la Práctica 3. Al ejecutar de nuevo el notebook, se vuelve a generar este derivado desde el CSV limpio y la misma fuente original, verificando su correspondencia.


In [43]:
# Guarda el DataFrame procesado.
RUTA_ENDIREH_RENOMBRADO_CSV.parent.mkdir(parents=True, exist_ok=True)
df_renombrado.write_csv(RUTA_ENDIREH_RENOMBRADO_CSV)

# Verifica que el archivo guardado coincida con el DataFrame original.
df_verificado = pl.read_csv(
    RUTA_ENDIREH_RENOMBRADO_CSV,
    schema_overrides=df_renombrado.schema
)

if not df_verificado.equals(df_renombrado):
    raise ValueError("El CSV generado no coincide con los datos preparados.")

print(f"Generado: {RUTA_ENDIREH_RENOMBRADO_CSV.name}")
print(f"Registros: {df_verificado.height:,} | Columnas: {df_verificado.width}")
print(f"Columnas renombradas: {len(columnas_renombradas)}")

df_verificado.head(5)


Generado: endireh_2021_renombrado.csv
Registros: 110,127 | Columnas: 24
Columnas renombradas: 11


cve_entidad,nom_entidad,cve_municipio,nom_municipio,anios_sin_convivencia_pareja,tipo_union,tipo_cuestionario,estado_civil_id,estado_civil_desc,estrato_sociodemografico_muestreo,entrevistada_trabaja_id,entrevistada_trabaja_desc,ingreso_laboral_entrevistada,dinero_propio_id,dinero_propio_desc,apoyo_gobierno_id,apoyo_gobierno_desc,hogar_tiene_ahorros_id,hogar_tiene_ahorros_desc,vivienda_propiedad_miembro_hogar_id,vivienda_propiedad_miembro_hogar_desc,sufrio_violencia_pareja,factor_expansion,anio_encuesta
i64,str,i64,str,i64,i64,str,i64,str,i64,i64,str,f64,i64,str,i64,str,i64,str,i64,str,i64,i64,i64
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",null,3,"""A1""",5,"""Casada""",4,1,"""Sí""",10000.0,1,"""Sí""",2,"""No""",2,"""No""",1,"""Sí""",0,113,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",null,3,"""A1""",5,"""Casada""",4,2,"""No""",null,2,"""No""",1,"""Sí""",2,"""No""",1,"""Sí""",1,113,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",13,3,"""B2""",4,"""Viuda""",4,2,"""No""",null,2,"""No""",1,"""Sí""",2,"""No""",2,"""No""",0,227,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",null,null,"""C1""",6,"""Soltera""",4,1,"""Sí""",8500.0,1,"""Sí""",2,"""No""",1,"""Sí""",1,"""Sí""",0,113,2021
1,"""AGUASCALIENTES""",1,"""AGUASCALIENTES""",20,3,"""B1""",2,"""Separada""",2,1,"""Sí""",400.0,1,"""Sí""",2,"""No""",2,"""No""",1,"""Sí""",1,155,2021


## 6. Decisiones heredadas y alcance

- Los montos de ingreso conservan sus valores recibidos, incluidos los codigos especiales. 
- `dinero_propio` describe disponibilidad de dinero que la entrevistada puede usar; `tipo_union` es una variable nominal, no un conteo; `tipo_cuestionario` tampoco es un nivel educativo.
- Se conserva el indicador `sufrio_violencia_pareja` proporcionado por el curso y los factores de expansion originales.
- No se imputan faltantes, eliminan filas ni recalculan los resultados de la Practica 3.
- El filtro de 10 años se corrige solo para este derivado: se recuperan los valores originales de 0 a 9, el codigo 0 conserva el significado «menos de un año».
- Las futuras ejecuciones de localizacion y variabilidad deben usar el CSV actualizado; sus resultados anteriores dejan de corresponder a esta entrada.